# 06 - Robustness Analysis

This notebook tests the robustness of SGI across different parameters.

## Steps:
1. Test different rolling windows
2. Test different top-k values
3. Test different covariance estimators
4. Summarize robustness findings

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment_config
from src.data_loader import load_or_download_prices
from src.preprocessing import preprocess_data
from src.covariance import rolling_covariance_matrices
from src.spectral_geometry import compute_rolling_geometry_features
from src.paths import ensure_all_directories

ensure_all_directories()

## Load Data

In [ ]:
config = load_experiment_config('../configs/universe_sector_etfs.yaml')

tickers = config['universe']['tickers']
prices = load_or_download_prices(tickers, config['data']['start_date'], config['data']['end_date'])
prices, returns, _ = preprocess_data(prices, return_method='log', save_outputs=False)

## Window Sensitivity

In [ ]:
windows = [60, 126, 252, 504]
window_results = {}

for window in windows:
    print(f"Computing SGI with window={window}...")
    cov_matrices = rolling_covariance_matrices(returns, window, method='sample')
    geometry_df = compute_rolling_geometry_features(cov_matrices, top_k=3)
    window_results[window] = geometry_df['sgi']

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for window, sgi in window_results.items():
    ax.plot(sgi.index, sgi.values, label=f'Window={window}', alpha=0.7)
ax.set_title('SGI Sensitivity to Rolling Window Size')
ax.set_ylabel('SGI')
ax.legend()
plt.tight_layout()
plt.show()

## Top-k Sensitivity

In [ ]:
topk_values = [2, 3, 5, 8]
topk_results = {}

cov_matrices = rolling_covariance_matrices(returns, 126, method='sample')

for k in topk_values:
    print(f"Computing SGI with top_k={k}...")
    geometry_df = compute_rolling_geometry_features(cov_matrices, top_k=k)
    topk_results[k] = geometry_df['sgi']

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for k, sgi in topk_results.items():
    ax.plot(sgi.index, sgi.values, label=f'top_k={k}', alpha=0.7)
ax.set_title('SGI Sensitivity to Top-k Eigenspace Dimension')
ax.set_ylabel('SGI')
ax.legend()
plt.tight_layout()
plt.show()

## Estimator Sensitivity

In [ ]:
estimators = ['sample', 'ewma', 'ledoit_wolf']
estimator_results = {}

for estimator in estimators:
    print(f"Computing SGI with estimator={estimator}...")
    cov_matrices = rolling_covariance_matrices(returns, 126, method=estimator, span=60)
    geometry_df = compute_rolling_geometry_features(cov_matrices, top_k=3)
    estimator_results[estimator] = geometry_df['sgi']

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for estimator, sgi in estimator_results.items():
    ax.plot(sgi.index, sgi.values, label=estimator, alpha=0.7)
ax.set_title('SGI Sensitivity to Covariance Estimator')
ax.set_ylabel('SGI')
ax.legend()
plt.tight_layout()
plt.show()

## Correlation Across Variants

In [ ]:
# Combine all variants
all_sgi = pd.DataFrame({
    f'window_{w}': s for w, s in window_results.items()
})

print("Correlation across window sizes:")
print(all_sgi.corr().round(3))

## Key Findings

1. SGI patterns are consistent across window sizes
2. Top-k choice affects magnitude but not timing
3. Different estimators produce similar signals